In [ ]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections

In [ ]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

**IMPORTANTE**: el catalogo `mensajeria_tiposervicio` (Administrativo/Comercial/Clinico/Urgencia Vital) NO es la fuente correcta para esta dimension --
esos valores describen el tipo de carga/documento, no la ventana de tiempo de entrega.

La fuente real es el campo de texto libre `mensajeria_servicio.prioridad`, que tiene variantes de formato
(mayúsculas/minúsculas, "a" vs "-") por lo que se agrupan por el prefijo Alta/Media/Baja en vez de comparar el texto exacto.

In [ ]:
prioridades_raw = pd.read_sql_query('''
    SELECT prioridad, count(*) as n
    FROM mensajeria_servicio
    WHERE prioridad IS NOT NULL AND trim(prioridad) <> ''
    GROUP BY prioridad
    ORDER BY n DESC
''', co_sa)
prioridades_raw

# Transformations

Se normaliza cada valor por su prefijo (antes de los dos puntos) y se mapea a un catalogo fijo de 3 categorias.

In [ ]:
def normalizar_prioridad(valor: str) -> str:
    prefijo = valor.split(':')[0].strip().lower()
    return prefijo  # 'alta', 'media' o 'baja'

prioridades_raw['grupo'] = prioridades_raw['prioridad'].apply(normalizar_prioridad)

# verificar que no aparezca ningun grupo inesperado (ej. error de tipeo nuevo)
grupos_esperados = {'alta', 'media', 'baja'}
grupos_encontrados = set(prioridades_raw['grupo'].unique())
if grupos_encontrados - grupos_esperados:
    print('ATENCION: hay grupos de prioridad no contemplados:', grupos_encontrados - grupos_esperados)

prioridades_raw

In [ ]:
catalogo_tipo_servicio = {
    'alta':  {'nombre_tipo': 'Urgente (menos de 1 hora)',
              'descripcion': 'Entrega prioritaria inmediata. Tiempo objetivo de entrega menor a 60 minutos desde la solicitud.'},
    'media': {'nombre_tipo': 'Media (1 a 3 horas)',
              'descripcion': 'Entrega en una ventana de 1 a 3 horas desde la solicitud.'},
    'baja':  {'nombre_tipo': 'Baja (transcurso del dia)',
              'descripcion': 'Entrega sin urgencia inmediata, dentro del transcurso del dia.'},
}

dim_tipo_servicio = pd.DataFrame([
    {'grupo': k, **v} for k, v in catalogo_tipo_servicio.items()
])
dim_tipo_servicio['saved'] = date.today()
dim_tipo_servicio

# Nota para quien construya el hecho Servicio

Para unir `mensajeria_servicio.prioridad` con esta dimension, **no comparar texto exacto** (hay variantes de formato).
Usar la misma logica de normalizacion por prefijo, por ejemplo en SQL:
```sql
CASE
    WHEN lower(split_part(prioridad, ':', 1)) = 'alta'  THEN 'alta'
    WHEN lower(split_part(prioridad, ':', 1)) = 'media' THEN 'media'
    WHEN lower(split_part(prioridad, ':', 1)) = 'baja'  THEN 'baja'
END AS grupo_prioridad
```
y luego hacer join de `grupo_prioridad` contra la columna `grupo` de `dim_tipo_servicio` (antes de cargarla, esa columna se puede dejar como atributo extra para facilitar este join futuro).

# load

In [ ]:
dim_tipo_servicio.to_sql('dim_tipo_servicio', etl_conn, if_exists='replace', index_label='key_dim_tipo_servicio')